# Late Fusion: Joint + Bone Skeleton Violence Detection (RWF-2000)

In this notebook, we combine the predictions of two independently trained skeleton-based models
using late fusion to improve violence detection accuracy on the RWF-2000 dataset.

## What is done
- Load pre-trained Joint stream model (2s-AGCN) from checkpoint
- Load pre-trained Bone stream model (2s-AGCN) from checkpoint
- Run Joint stream inference using MMAction2's official test pipeline (10-clip TTA)
- Run Bone stream inference using MMAction2's official test pipeline (10-clip TTA)
- Ensure both streams are evaluated on the validation set in the exact same order
- Fuse both models' softmax probabilities with different weights
- Evaluate fusion accuracy and F1 score

## Models
| Model | Architecture | Individual Accuracy |
|-------|-------------|-------------------|
| Joint Stream | 2s-AGCN (joint features) | 80.75% |
| Bone Stream | 2s-AGCN (bone features) | 83.50% |

## Fusion Method
Late fusion — weighted average of softmax probabilities:
```
fused_score = w_joint × P_joint + w_bone × P_bone
prediction  = argmax(fused_score)
```

No retraining required.

## Dataset
RWF-2000 validation set
400 videos (200 Fight, 200 NonFight)
Clean split with 0 overlap between train and val

## Checkpoints Used
Joint: checkpoints/pose/2s-agcn-joint/best_acc_top1_epoch_8.pth
Bone: checkpoints/pose/2s-agcn-bone/best_acc_top1_epoch_13.pth

## Output
Best fusion result:
Equal weighting (0.5 + 0.5) → 84.00% accuracy, F1: 0.8351




## Imports and installations

In [ ]:
# Mount Google Drive to access our saved files
from google.colab import drive
drive.mount('/content/drive')

# Install mmcv from saved wheel (fast, avoids 40min compilation)
!pip install /content/drive/MyDrive/mmcv-2.1.0-cp312-cp312-linux_x86_64.whl -q
!pip install mmengine -q

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.7/452.7 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 28.3 MB/s eta 0:00:00


In [ ]:
# Clone MMAction2 repository and install it
# We disable the multimodal module since we don't need it
# and it causes import errors with newer transformers
%cd /content
!git clone https://github.com/open-mmlab/mmaction2.git -q
%cd /content/mmaction2
!pip install -e . -q
!sed -i 's/from .multimodal import \*/# from .multimodal import */' \
    /content/mmaction2/mmaction/models/__init__.py

/content
fatal: destination path 'mmaction2' already exists and is not an empty directory.
/content/mmaction2
  Preparing metadata (setup.py) ... done


In [ ]:
# PyTorch 2.10 changed how it loads checkpoints by default
# MMAction2 wasn't updated for this change, so we patch it manually
!sed -i 's/checkpoint = torch.load(filename, map_location=map_location)/checkpoint = torch.load(filename, map_location=map_location, weights_only=False)/' \
    /usr/local/lib/python3.12/dist-packages/mmengine/runner/checkpoint.py
print("Patched!")

Patched!


In [ ]:
# After installing new libraries, Python needs a fresh start
# to recognize them properly
import os
os.kill(os.getpid(), 9)

In [ ]:
      # Load all required libraries
import torch
import torch.nn as nn
import numpy as np
import pickle
import os
from glob import glob
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import r3d_18
from decord import VideoReader, cpu
import torchvision.transforms as T
from sklearn.metrics import accuracy_score, f1_score, classification_report

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda


## Paths

In [ ]:
# ========= PATHS =========
MMACTION_ROOT = '/content/mmaction2'

ANN_FILE = '/content/drive/MyDrive/violence-detection/data/processed/pose/pkl/rwf_pose.pkl'

JOINT_CKPT = '/content/drive/MyDrive/violence-detection/checkpoints/pose/2s-agcn-joint/best_acc_top1_epoch_8.pth'
BONE_CKPT  = '/content/drive/MyDrive/violence-detection/checkpoints/pose/2s-agcn-bone/best_acc_top1_epoch_13.pth'

JOINT_CONFIG_PATH = f'{MMACTION_ROOT}/configs/skeleton/2s-agcn/2s-agcn_joint_test.py'
BONE_CONFIG_PATH  = f'{MMACTION_ROOT}/configs/skeleton/2s-agcn/2s-agcn_bone_test.py'

JOINT_DUMP_LOCAL = '/content/joint_predictions.pkl'
BONE_DUMP_LOCAL  = '/content/bone_predictions.pkl'

JOINT_DUMP_DRIVE = '/content/drive/MyDrive/violence-detection/checkpoints/pose/2s-agcn-joint/joint_val_predictions.pkl'
BONE_DUMP_DRIVE  = '/content/drive/MyDrive/violence-detection/checkpoints/pose/2s-agcn-bone/bone_val_predictions.pkl'

print("Paths set.")

Paths set.


## Joint Test Config

In [ ]:
joint_test_config = f"""
_base_ = '../../_base_/default_runtime.py'

model = dict(
    type='RecognizerGCN',
    backbone=dict(
        type='AAGCN',
        graph_cfg=dict(layout='coco', mode='spatial'),
        gcn_attention=False),
    cls_head=dict(type='GCNHead', num_classes=2, in_channels=256))

dataset_type = 'PoseDataset'
ann_file = '{ANN_FILE}'

test_pipeline = [
    dict(type='PreNormalize2D'),
    dict(type='GenSkeFeat', dataset='coco', feats=['j']),
    dict(type='UniformSampleFrames', clip_len=100, num_clips=10, test_mode=True),
    dict(type='PoseDecode'),
    dict(type='FormatGCNInput', num_person=2),
    dict(type='PackActionInputs')
]

test_dataloader = dict(
    batch_size=1,
    num_workers=2,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type=dataset_type,
        ann_file=ann_file,
        pipeline=test_pipeline,
        split='val',
        test_mode=True))

test_evaluator = [dict(type='AccMetric')]
test_cfg = dict(type='TestLoop')
"""

with open(JOINT_CONFIG_PATH, 'w') as f:
    f.write(joint_test_config)

print("Joint test config saved:", JOINT_CONFIG_PATH)

Joint test config saved: /content/mmaction2/configs/skeleton/2s-agcn/2s-agcn_joint_test.py


In [ ]:
%cd /content/mmaction2

!python tools/test.py \
    configs/skeleton/2s-agcn/2s-agcn_joint_test.py \
    "$JOINT_CKPT" \
    --dump "$JOINT_DUMP_LOCAL"

/content/mmaction2
04/03 08:08:30 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 197064223
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.10.0+cu128
    PyTorch compiling details: PyTorch built with:
  - GCC 13.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2024.2-Product Build 20240605 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.7.1 (Git Hash 8d263e693366ef8db40acc569cc7d8edf644556d)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 12.8
  - NVCC architecture flags: -gencode;

In [ ]:
import shutil

shutil.copy(JOINT_DUMP_LOCAL, JOINT_DUMP_DRIVE)
print("Joint predictions saved to Drive:", JOINT_DUMP_DRIVE)

Joint predictions saved to Drive: /content/drive/MyDrive/violence-detection/checkpoints/pose/2s-agcn-joint/joint_val_predictions.pkl


## Stream Test Config

In [ ]:
bone_test_config = f"""
_base_ = '../../_base_/default_runtime.py'

model = dict(
    type='RecognizerGCN',
    backbone=dict(
        type='AAGCN',
        graph_cfg=dict(layout='coco', mode='spatial'),
        gcn_attention=False),
    cls_head=dict(type='GCNHead', num_classes=2, in_channels=256))

dataset_type = 'PoseDataset'
ann_file = '{ANN_FILE}'

test_pipeline = [
    dict(type='PreNormalize2D'),
    dict(type='GenSkeFeat', dataset='coco', feats=['b']),
    dict(type='UniformSampleFrames', clip_len=100, num_clips=10, test_mode=True),
    dict(type='PoseDecode'),
    dict(type='FormatGCNInput', num_person=2),
    dict(type='PackActionInputs')
]

test_dataloader = dict(
    batch_size=1,
    num_workers=2,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type=dataset_type,
        ann_file=ann_file,
        pipeline=test_pipeline,
        split='val',
        test_mode=True))

test_evaluator = [dict(type='AccMetric')]
test_cfg = dict(type='TestLoop')
"""

with open(BONE_CONFIG_PATH, 'w') as f:
    f.write(bone_test_config)

print("Bone test config saved:", BONE_CONFIG_PATH)

Bone test config saved: /content/mmaction2/configs/skeleton/2s-agcn/2s-agcn_bone_test.py


In [ ]:
%cd /content/mmaction2

!python tools/test.py \
    configs/skeleton/2s-agcn/2s-agcn_bone_test.py \
    "$BONE_CKPT" \
    --dump "$BONE_DUMP_LOCAL"

/content/mmaction2
04/03 08:10:32 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1150984504
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.10.0+cu128
    PyTorch compiling details: PyTorch built with:
  - GCC 13.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2024.2-Product Build 20240605 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.7.1 (Git Hash 8d263e693366ef8db40acc569cc7d8edf644556d)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 12.8
  - NVCC architecture flags: -gencode

In [ ]:
shutil.copy(BONE_DUMP_LOCAL, BONE_DUMP_DRIVE)
print("Bone predictions saved to Drive:", BONE_DUMP_DRIVE)

Bone predictions saved to Drive: /content/drive/MyDrive/violence-detection/checkpoints/pose/2s-agcn-bone/bone_val_predictions.pkl


## Load Joint and Bone Predictions

In [ ]:
with open(JOINT_DUMP_DRIVE, 'rb') as f:
    joint_preds = pickle.load(f)

joint_probs = np.array([p['pred_score'].numpy() for p in joint_preds])
joint_labels = np.array([p['gt_label'].item() for p in joint_preds])

print("Joint probs shape:", joint_probs.shape)
print("Joint labels shape:", joint_labels.shape)
print("Joint accuracy:", accuracy_score(joint_labels, np.argmax(joint_probs, axis=1)))

Joint probs shape: (400, 2)
Joint labels shape: (400,)
Joint accuracy: 0.8075


In [ ]:
with open(BONE_DUMP_DRIVE, 'rb') as f:
    bone_preds = pickle.load(f)

bone_probs = np.array([p['pred_score'].numpy() for p in bone_preds])
bone_labels = np.array([p['gt_label'].item() for p in bone_preds])

print("Bone probs shape:", bone_probs.shape)
print("Bone labels shape:", bone_labels.shape)
print("Bone accuracy:", accuracy_score(bone_labels, np.argmax(bone_probs, axis=1)))

Bone probs shape: (400, 2)
Bone labels shape: (400,)
Bone accuracy: 0.835


## Safety Check: orders match?

In [ ]:
print("Same number of samples:", len(joint_labels) == len(bone_labels))
print("Labels match:", np.all(joint_labels == bone_labels))

assert len(joint_labels) == len(bone_labels), "Different number of validation samples!"
assert np.all(joint_labels == bone_labels), "Joint and Bone prediction order DOES NOT match!"

labels = joint_labels
print("Safety checks passed.")

Same number of samples: True
Labels match: True
Safety checks passed.


## Fusion

In [ ]:
print("=" * 60)
print(f"{'Fusion Weights':<30} {'Accuracy':>10} {'F1':>10}")
print("=" * 60)

best_acc = -1
best_f1 = -1
best_preds = None
best_weights = None
best_fused_probs = None

for w_joint in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    w_bone = 1.0 - w_joint

    fused_probs = w_joint * joint_probs + w_bone * bone_probs
    preds = np.argmax(fused_probs, axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)

    print(f"Joint {w_joint:.1f} + Bone {w_bone:.1f}{'':<8} {acc:>10.4f} {f1:>10.4f}")

    if acc > best_acc or (acc == best_acc and f1 > best_f1):
        best_acc = acc
        best_f1 = f1
        best_preds = preds
        best_weights = (w_joint, w_bone)
        best_fused_probs = fused_probs

print("=" * 60)
print(f"{'Joint only':<30} {accuracy_score(labels, np.argmax(joint_probs, axis=1)):>10.4f}")
print(f"{'Bone only':<30} {accuracy_score(labels, np.argmax(bone_probs, axis=1)):>10.4f}")
print(f"{'Best fusion':<30} {best_acc:>10.4f}")
print(f"Best weights: Joint {best_weights[0]:.1f} + Bone {best_weights[1]:.1f}")

Fusion Weights                   Accuracy         F1
Joint 0.0 + Bone 1.0             0.8350     0.8272
Joint 0.1 + Bone 0.9             0.8325     0.8241
Joint 0.2 + Bone 0.8             0.8350     0.8272
Joint 0.3 + Bone 0.7             0.8350     0.8281
Joint 0.4 + Bone 0.6             0.8375     0.8303
Joint 0.5 + Bone 0.5             0.8400     0.8351
Joint 0.6 + Bone 0.4             0.8375     0.8338
Joint 0.7 + Bone 0.3             0.8250     0.8250
Joint 0.8 + Bone 0.2             0.8225     0.8229
Joint 0.9 + Bone 0.1             0.8100     0.8109
Joint 1.0 + Bone 0.0             0.8075     0.8089
Joint only                         0.8075
Bone only                          0.8350
Best fusion                        0.8400
Best weights: Joint 0.5 + Bone 0.5


## Analysis

In [ ]:
print("=== Best Joint + Bone Fusion ===")
print(f"Best weights: Joint {best_weights[0]:.1f} + Bone {best_weights[1]:.1f}")
print()
print(classification_report(labels, best_preds, target_names=['NonFight', 'Fight']))

=== Best Joint + Bone Fusion ===
Best weights: Joint 0.5 + Bone 0.5

              precision    recall  f1-score   support

    NonFight       0.82      0.87      0.84       200
       Fight       0.86      0.81      0.84       200

    accuracy                           0.84       400
   macro avg       0.84      0.84      0.84       400
weighted avg       0.84      0.84      0.84       400



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels, best_preds)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[174  26]
 [ 38 162]]


In [ ]:
fusion_results = {
    'labels': labels,
    'joint_probs': joint_probs,
    'bone_probs': bone_probs,
    'best_fused_probs': best_fused_probs,
    'best_preds': best_preds,
    'best_weights': best_weights,
    'best_acc': best_acc,
    'best_f1': best_f1,
}

save_path = '/content/drive/MyDrive/violence-detection/checkpoints/pose/joint_bone_fusion_results.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(fusion_results, f)

print("Fusion results saved to:", save_path)

Fusion results saved to: /content/drive/MyDrive/violence-detection/checkpoints/pose/joint_bone_fusion_results.pkl


In [ ]:
joint_acc = accuracy_score(labels, np.argmax(joint_probs, axis=1))
bone_acc = accuracy_score(labels, np.argmax(bone_probs, axis=1))

print("\n===== FINAL SUMMARY =====")
print(f"Joint only acc: {joint_acc:.4f}")
print(f"Bone only acc : {bone_acc:.4f}")
print(f"Best fusion acc: {best_acc:.4f}")
print(f"Best fusion f1 : {best_f1:.4f}")
print(f"Best weights   : Joint {best_weights[0]:.1f} + Bone {best_weights[1]:.1f}")


===== FINAL SUMMARY =====
Joint only acc: 0.8075
Bone only acc : 0.8350
Best fusion acc: 0.8400
Best fusion f1 : 0.8351
Best weights   : Joint 0.5 + Bone 0.5
